In [1]:
import openeo, json, os
import geopandas as gpd

# 1. Read GeoJSON Dataset (input2.geojson)
gdf = gpd.read_file("../data/geojson/input2.geojson")
print("=== Jumlah Polygon per Kelas ===")
print(gdf["Type"].value_counts())

# Ambil Poligon Batas Daerah sebagai Spatial Extent
daerah = gdf[gdf["Type"] == "Daerah"].geometry.iloc[0]
geom = json.loads(gpd.GeoSeries([daerah]).to_json())["features"][0]["geometry"]

# 2. Koneksi ke Backend openEO Copernicus Data Space Ecosystem
try:
    conn = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
    bands = ["B02", "B03", "B04", "B08", "B11", "SCL"]
    cube = conn.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=geom,
        temporal_extent=["2025-08-24", "2026-08-23"],
        bands=bands,
        max_cloud_cover=30,
    )
    scl = cube.band("SCL")
    mask = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)
    cube = cube.mask(mask.resample_cube_spatial(cube)).filter_bands(bands[:-1])
    cube = cube.reduce_dimension(dimension="t", reducer="median")
    os.makedirs("../data/tif", exist_ok=True)
    cube.download("../data/tif/sentinel2_area_studi.tif", format="GTiff")
    print("SUCCESS: Berhasil mengunduh citra Sentinel-2A via openEO.")
except Exception as e:
    print(f"[openEO Info]: {e}")
    print("Memastikan file raster GeoTIFF lokal tersedia di '../data/tif/sentinel2_area_studi.tif'...")

=== Jumlah Polygon per Kelas ===
Type
Sawah        50
Pemukiman    50
Daerah        1
Name: count, dtype: int64


Authenticated using refresh token.


SUCCESS: Berhasil mengunduh citra Sentinel-2A via openEO.


In [2]:
import os, rasterio, numpy as np, pandas as pd
from rasterio.mask import mask as rio_mask

samples = gdf[gdf["Type"].isin(["Sawah", "Pemukiman"])].copy()
tif_path = "../data/tif/sentinel2_area_studi.tif"

if os.path.exists(tif_path):
    with rasterio.open(tif_path) as src:
        samples = samples.to_crs(src.crs)
        rows = []
        for idx, r in samples.iterrows():
            try:
                m, _ = rio_mask(src, [r.geometry], crop=True, all_touched=True, filled=False)
            except ValueError:
                continue
            arr = m.astype("float32").filled(np.nan) / 10000
            if src.nodata is not None:
                arr[m.data == src.nodata] = np.nan
            b02, b03, b04, b08, b11 = [arr[k].ravel() for k in range(5)]
            ok = ~np.isnan(b04) & ~np.isnan(b08) & ~np.isnan(b03) & ~np.isnan(b11)
            if ok.sum() == 0:
                continue
            b02, b03, b04, b08, b11 = b02[ok], b03[ok], b04[ok], b08[ok], b11[ok]
            ndvi = (b08 - b04) / (b08 + b04 + 1e-6)
            ndwi = (b03 - b08) / (b03 + b08 + 1e-6)
            ndbi = (b11 - b08) / (b11 + b08 + 1e-6)
            rows.append({"id": idx, "kelas": r["Type"], "n_piksel": int(ok.sum()),
                         "B02": b02.mean(), "B03": b03.mean(), "B04": b04.mean(),
                         "B08": b08.mean(), "B11": b11.mean(),
                         "NDVI": ndvi.mean(), "NDWI": ndwi.mean(), "NDBI": ndbi.mean()})
    df = pd.DataFrame(rows)
    print(f"Total Sampel Poligon Valid: {len(df)}")
    print("\n=== Rata-rata Indeks Spektral per Kelas ===")
    print(df.groupby("kelas")[["NDVI", "NDWI", "NDBI"]].mean())
    os.makedirs("../data/csv", exist_ok=True)
    df.to_csv("../data/csv/fitur_sampel.csv", index=False)

Total Sampel Poligon Valid: 100

=== Rata-rata Indeks Spektral per Kelas ===
               NDVI      NDWI      NDBI
kelas                                  
Pemukiman  0.289784 -0.328458  0.079627
Sawah      0.340569 -0.235186 -0.263561


In [3]:
import rasterio, numpy as np
from rasterio.mask import mask as rio_mask
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score, classification_report

TIF = "../data/tif/sentinel2_area_studi.tif"
NAMA_FITUR = ["B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI", "NDBI"]

def hitung_fitur(a):
    b02, b03, b04, b08, b11 = a
    e = 1e-6
    ndvi = (b08 - b04) / (b08 + b04 + e)
    ndwi = (b03 - b08) / (b03 + b08 + e)
    ndbi = (b11 - b08) / (b11 + b08 + e)
    return np.stack([b02, b03, b04, b08, b11, ndvi, ndwi, ndbi], axis=-1)

if os.path.exists(TIF):
    X, y, grp = [], [], []
    with rasterio.open(TIF) as src:
        smp = samples.to_crs(src.crs)
        for idx, r in smp.iterrows():
            try:
                m, _ = rio_mask(src, [r.geometry], crop=True, all_touched=True, filled=False)
            except ValueError:
                continue
            a = m.astype("float32").filled(np.nan) / 10000
            f = hitung_fitur(a).reshape(-1, 8)
            f = f[~np.isnan(f).any(axis=1)]
            f = f[(f[:, :5] > 0).all(axis=1)]
            X.append(f); y += [r["Type"]] * len(f); grp += [idx] * len(f)
    X, y, grp = np.vstack(X), np.array(y), np.array(grp)
    print(f"Total Piksel Ekstraksi: {len(y)}")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr, te = next(gss.split(X, y, grp))
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X[tr], y[tr])
    pred = rf.predict(X[te])
    labels = ["Sawah", "Pemukiman"]
    print("\n=== CONFUSION MATRIX ===")
    print(confusion_matrix(y[te], pred, labels=labels))
    print(f"Overall Accuracy : {accuracy_score(y[te], pred) * 100:.2f}%")
    print(f"Cohen's Kappa    : {cohen_kappa_score(y[te], pred):.4f}")
    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y[te], pred))
    print("\n=== FEATURE IMPORTANCES ===")
    for n, v in sorted(zip(NAMA_FITUR, rf.feature_importances_), key=lambda t: -t[1]):
        print(f"{n:5s} : {v:.3f}")

Total Piksel Ekstraksi: 12037



=== CONFUSION MATRIX ===
[[ 885   15]
 [   0 1719]]
Overall Accuracy : 99.43%
Cohen's Kappa    : 0.9873

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

   Pemukiman       0.99      1.00      1.00      1719
       Sawah       1.00      0.98      0.99       900

    accuracy                           0.99      2619
   macro avg       1.00      0.99      0.99      2619
weighted avg       0.99      0.99      0.99      2619


=== FEATURE IMPORTANCES ===
B11   : 0.372
NDBI  : 0.210
B04   : 0.163
B08   : 0.107
B02   : 0.074
NDWI  : 0.034
B03   : 0.023
NDVI  : 0.017


In [4]:
import os, rasterio, numpy as np
import plotly.graph_objects as go
from rasterio.warp import reproject, Resampling
from IPython.display import IFrame, display

os.makedirs("../assets/editor/html", exist_ok=True)
html_path = "../assets/editor/html/peta_3d.html"
ref_tif = "../data/tif/klasifikasi_sawah_pemukiman.tif"
dem_tif = "../data/tif/dem_area_studi.tif"

if os.path.exists(ref_tif) and os.path.exists(dem_tif):
    with rasterio.open(ref_tif) as ref, rasterio.open(dem_tif) as dem:
        peta = ref.read(1).astype("float32")
        res = ref.res[0]
        dem_arr = np.zeros(peta.shape, dtype="float32")
        reproject(source=rasterio.band(dem, 1), destination=dem_arr,
                  dst_transform=ref.transform, dst_crs=ref.crs,
                  resampling=Resampling.bilinear)
    peta[peta == 0] = np.nan
    STEP = 3
    Z = dem_arr[::STEP, ::STEP]
    C = peta[::STEP, ::STEP]
    x = np.arange(Z.shape[1]) * res * STEP
    y = np.arange(Z.shape[0]) * res * STEP
    colorscale = [[0, "#2e8b57"], [0.5, "#2e8b57"], [0.5, "#d2691e"], [1, "#d2691e"]]
    fig = go.Figure(go.Surface(
        x=x, y=y, z=Z, surfacecolor=C,
        colorscale=colorscale, cmin=1, cmax=2,
        colorbar=dict(title="Kelas", tickvals=[1.25, 1.75], ticktext=["Sawah", "Pemukiman"]),
        lighting=dict(ambient=0.6, diffuse=0.8, roughness=0.9),
    ))
    fig.update_layout(
        title="Klasifikasi Sawah & Pemukiman (3D Surface Relief)",
        scene=dict(aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.25),
                   xaxis_title="Timur (m)", yaxis_title="Utara (m)", zaxis_title="Elevasi (m)"),
        height=750,
    )
    fig.write_html(html_path)
    print(f"SUCCESS: Peta 3D berhasil disimpan di '{html_path}'")
    display(IFrame(html_path, width="100%", height=750))

SUCCESS: Peta 3D berhasil disimpan di '../assets/editor/html/peta_3d.html'
